### Restart and Run All Cells

In [2]:
import pandas as pd
from datetime import date, timedelta, datetime
from sqlalchemy import create_engine, text

engine = create_engine("sqlite:///c:\\ruby\\portlt\\db\\development.sqlite3")
conlt = engine.connect()

year = 2026
quarter = 1
current_time = datetime.now()
formatted_time = current_time.strftime("%Y:%m:%d %H:%M:%S")
print(formatted_time)

2026:05:17 14:52:15


In [3]:
cols = 'name year quarter q_amt_c q_amt_p inc_profit percent'.split()

format_dict = {
                'q_amt':'{:,}','q_amt_c':'{:,}','q_amt_p':'{:,}','inc_profit':'{:,}',
                'yoy_gain':'{:,}','acc_gain':'{:,}',    
                'q_eps':'{:.4f}','y_eps':'{:.4f}','aq_eps':'{:.4f}','ay_eps':'{:.4f}',
                'percent':'{:.2f}%'
              }

In [4]:
sql = """
SELECT name,year,quarter,q_amt
FROM epss 
WHERE (year = %s AND quarter <= %s) 
OR (year = %s-1 AND quarter >= %s+1)
ORDER BY year DESC, quarter DESC"""
sql = sql % (year, quarter, year, quarter)
print(sql)


SELECT name,year,quarter,q_amt
FROM epss 
WHERE (year = 2026 AND quarter <= 1) 
OR (year = 2026-1 AND quarter >= 1+1)
ORDER BY year DESC, quarter DESC


In [5]:
dfc = pd.read_sql(sql, conlt)
dfc["Counter"] = 1
dfc_grp = dfc.groupby(["name"], as_index=False).sum()
dfc_grp = dfc_grp[dfc_grp["Counter"] == 4]
dfc_grp.head().style.format(format_dict)

,name,year,quarter,q_amt,Counter
0,3BBIF,8101,10,"6,831,513",4
1,ACE,8101,10,"886,861",4
2,ADVANC,8101,10,"50,797,881",4
4,AH,8101,10,"739,606",4
5,AIE,8101,10,"62,999",4


In [6]:
sql = """
SELECT name,year,quarter,q_amt
FROM epss 
WHERE (year = %s-1 AND quarter <= %s - 1) 
OR (year = %s-2 AND quarter >= %s)
ORDER BY year DESC, quarter DESC
"""
sql = sql % (year, quarter, year, quarter)
print(sql)


SELECT name,year,quarter,q_amt
FROM epss 
WHERE (year = 2026-1 AND quarter <= 1 - 1) 
OR (year = 2026-2 AND quarter >= 1)
ORDER BY year DESC, quarter DESC



In [7]:
dfp = pd.read_sql(sql, conlt)
dfp["Counter"] = 1
dfp_grp = dfp.groupby(["name"], as_index=False).sum()
dfp_grp = dfp_grp[dfp_grp["Counter"] == 4]
dfp_grp.head().style.format(format_dict)

,name,year,quarter,q_amt,Counter
0,3BBIF,8096,10,"5,279,119",4
1,ACE,8096,10,"838,717",4
2,ADVANC,8096,10,"35,075,357",4
3,AEONTS,8096,10,"2,860,344",4
4,AH,8096,10,"746,961",4


In [8]:
dfm = pd.merge(dfc_grp, dfp_grp, on="name", suffixes=(["_c", "_p"]), how="inner")
dfm["inc_profit"] = dfm["q_amt_c"] - dfm["q_amt_p"]
dfm["percent"] = round(dfm["inc_profit"] / abs(dfm["q_amt_p"]) * 100, 2)
dfm["year"] = year
dfm["quarter"] = "Q" + str(quarter)
df_percent = dfm[cols]
df_percent.head().style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent
0,3BBIF,2026,Q1,"6,831,513","5,279,119","1,552,394",29.41%
1,ACE,2026,Q1,"886,861","838,717","48,144",5.74%
2,ADVANC,2026,Q1,"50,797,881","35,075,357","15,722,524",44.82%
3,AH,2026,Q1,"739,606","746,961","-7,355",-0.98%
4,AIE,2026,Q1,"62,999","241,922","-178,923",-73.96%


In [9]:
# Create the SQL query with parameter binding
sql = text("DELETE FROM yr_profits WHERE year = :year AND quarter = :quarter")

# Execute the query with parameters
params = {'year': year, 'quarter': f'Q{quarter}'}
rp = conlt.execute(sql, params)

# Print the number of rows affected
print("Rows deleted:", rp.rowcount)

Rows deleted: 54


In [10]:
sql = "SELECT name, id FROM tickers"
tickers = pd.read_sql(sql, conlt)
df_ins = pd.merge(df_percent, tickers, on="name", how="inner")
rcds = df_ins.values.tolist()
len(rcds)

181

In [11]:
# Convert DataFrame to list of records
rcds = df_ins.values.tolist()

# Define column names in the same order as values
columns = ['name', 'year', 'quarter', 'latest_amt', 'previous_amt', 'inc_amt', 'inc_pct', 'ticker_id']

# SQL insert statement with named parameters
sql = text("""
    INSERT INTO yr_profits 
    (name, year, quarter, latest_amt, previous_amt, inc_amt, inc_pct, ticker_id)
    VALUES (:name, :year, :quarter, :latest_amt, :previous_amt, :inc_amt, :inc_pct, :ticker_id)
""")

try:
    # Execute inserts
    for rcd in rcds:
        # Convert list to dictionary
        params = dict(zip(columns, rcd))
        conlt.execute(sql, params)
except Exception as e:
    raise e

### End of loop

In [13]:
criteria_1 = df_ins.q_amt_c > 440_000
df_ins.loc[criteria_1, cols].head().style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent
0,3BBIF,2026,Q1,"6,831,513","5,279,119","1,552,394",29.41%
1,ACE,2026,Q1,"886,861","838,717","48,144",5.74%
2,ADVANC,2026,Q1,"50,797,881","35,075,357","15,722,524",44.82%
3,AH,2026,Q1,"739,606","746,961","-7,355",-0.98%
5,AIMIRT,2026,Q1,"702,550","948,696","-246,146",-25.95%


In [14]:
criteria_2 = df_ins.q_amt_p > 400_000
df_ins.loc[criteria_2, cols].head().style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent
0,3BBIF,2026,Q1,"6,831,513","5,279,119","1,552,394",29.41%
1,ACE,2026,Q1,"886,861","838,717","48,144",5.74%
2,ADVANC,2026,Q1,"50,797,881","35,075,357","15,722,524",44.82%
3,AH,2026,Q1,"739,606","746,961","-7,355",-0.98%
5,AIMIRT,2026,Q1,"702,550","948,696","-246,146",-25.95%


In [15]:
criteria_3 = df_ins.percent > 10.00
df_ins.loc[criteria_3, cols].head().style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent
0,3BBIF,2026,Q1,"6,831,513","5,279,119","1,552,394",29.41%
2,ADVANC,2026,Q1,"50,797,881","35,075,357","15,722,524",44.82%
7,AMATA,2026,Q1,"3,697,994","2,482,899","1,215,095",48.94%
12,ASK,2026,Q1,"587,609","331,797","255,812",77.10%
17,BAM,2026,Q1,"1,812,151","1,601,642","210,509",13.14%


In [16]:
final_criteria = criteria_1 & criteria_2 & criteria_3
df_ins.loc[final_criteria, cols].sort_values(by=["percent"], ascending=[False]).head().style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent
50,DIF,2026,Q1,"13,891,180","656,288","13,234,892",2016.63%
61,GULF,2026,Q1,"89,114,570","18,170,334","70,944,236",390.44%
33,BPP,2026,Q1,"8,329,005","1,746,321","6,582,684",376.95%
139,SPRC,2026,Q1,"9,222,759","2,234,886","6,987,873",312.67%
21,BCP,2026,Q1,"6,908,168","2,184,088","4,724,080",216.30%


In [17]:
df_ins.loc[final_criteria, cols].sort_values(by=["name"], ascending=[True]).head().style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent
0,3BBIF,2026,Q1,"6,831,513","5,279,119","1,552,394",29.41%
2,ADVANC,2026,Q1,"50,797,881","35,075,357","15,722,524",44.82%
7,AMATA,2026,Q1,"3,697,994","2,482,899","1,215,095",48.94%
17,BAM,2026,Q1,"1,812,151","1,601,642","210,509",13.14%
21,BCP,2026,Q1,"6,908,168","2,184,088","4,724,080",216.30%


In [18]:
df_ins.loc[final_criteria, cols].sort_values(by=["name"], ascending=[True]).head().style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent
0,3BBIF,2026,Q1,"6,831,513","5,279,119","1,552,394",29.41%
2,ADVANC,2026,Q1,"50,797,881","35,075,357","15,722,524",44.82%
7,AMATA,2026,Q1,"3,697,994","2,482,899","1,215,095",48.94%
17,BAM,2026,Q1,"1,812,151","1,601,642","210,509",13.14%
21,BCP,2026,Q1,"6,908,168","2,184,088","4,724,080",216.30%


In [19]:
conlt.commit()
conlt.close()

In [20]:
current_time = datetime.now()
formatted_time = current_time.strftime("%Y:%m:%d %H:%M:%S")
print(formatted_time)

2026:05:17 14:52:16
